# Track C — Data Quality Check & Scaler Preparation
**Author:** Mohammed Ghannam Alyami  
**Project:** VOLTA — Energy Pressure Forecasting  
**Goal:** Conduct rigorous data quality checks, handle missing values, and prepare leak-free feature scaling on the training split.

In [1]:
# استيراد مكتبات التعامل مع البيانات والتحجيم
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

print('Track C Libraries Imported Successfully!')

Track C Libraries Imported Successfully!


## 1. Load Data & Verify Timeline Completeness
Load the dataset and sort chronologically to maintain time-series integrity.

In [2]:
# 1. قراءة البيانات المدمجة والمنظفة
df = pd.read_csv('../data/processed/volta_clean_hourly.csv')

# 2. ضبط التوقيت الزمني
df['time'] = pd.to_datetime(df['time'], utc=True)
df = df.sort_values('time').set_index('time')

# 3. عرض ملخص النقص لـ الأهداف الأربعة المعتمدة
print(f'Total Hourly Rows: {len(df)}')
print('Missing values in target columns:')
print(
    df[[
        'price actual',
        'total load actual',
        'renewable_total',
        'fossil_total',
    ]]
    .isnull()
    .sum()
)

Total Hourly Rows: 35064
Missing values in target columns:
price actual         0
total load actual    0
renewable_total      0
fossil_total         0
dtype: int64


## 2. Handle Missing Values
Apply forward filling (`ffill`) for small gaps in time-series data to ensure continuity without using future information.

In [3]:
# 1. تحديد أعمدة الأهداف الرئيسية
features_to_scale = [
    'price actual',
    'total load actual',
    'renewable_total',
    'fossil_total',
]

# 2. أخذ نسخة ومعالجة القيم المفقودة للأعمدة المستهدفة فقط
df_cleaned = df[features_to_scale].copy()
df_cleaned = df_cleaned.ffill().bfill()

# 3. التأكد من اختفاء جميع القيم المفقودة تماماً)
print(
    'Remaining Missing Values in Target Features:',
    df_cleaned.isnull().sum().sum(),
)

Remaining Missing Values in Target Features: 0


## 3. Chronological Train / Test Split
Split the data chronologically into:
- **Train Set:** 2015-01-01 to 2016-12-31
- **Validation set:** 2017-01-01 to 2017-12-31
- **Test Set:** 2018-01-01 to 2018-12-31 (Kept strictly untouched)

In [6]:
# 1. التقسيم الزمني المعياري الخالي من التسريب (2015-2016 تدريب | 2017 تحقق | 2018 اختبار)
train_data = df_cleaned.loc['2015-01-01':'2016-12-31']
val_data   = df_cleaned.loc['2017-01-01':'2017-12-31']
test_data  = df_cleaned.loc['2018-01-01':'2018-12-31']

print(f"Train Period:      {train_data.index.min()} to {train_data.index.max()} (Rows: {len(train_data)})")
print(f"Validation Period: {val_data.index.min()} to {val_data.index.max()} (Rows: {len(val_data)})")
print(f"Test Period:       {test_data.index.min()} to {test_data.index.max()} (Rows: {len(test_data)})")

Train Period:      2015-01-01 00:00:00+00:00 to 2016-12-31 23:00:00+00:00 (Rows: 17544)
Validation Period: 2017-01-01 00:00:00+00:00 to 2017-12-31 23:00:00+00:00 (Rows: 8760)
Test Period:       2018-01-01 00:00:00+00:00 to 2018-12-31 22:00:00+00:00 (Rows: 8759)


## 4. Leak-Free Feature Scaling
Fit the `MinMaxScaler` **strictly on the training data**, then transform both train and test sets independently.

In [8]:
# 1. إنشاء أداة التحجيم
scaler = MinMaxScaler(feature_range=(0, 1))

# 2. ملاءمة الـ Scaler على بيانات التدريب (2015-2016) فقط لمنع التسريب
train_scaled = scaler.fit_transform(train_data[features_to_scale])

# 3. تحويل بيانات التحقق والاختبار بنفس أداة التدريب
val_scaled  = scaler.transform(val_data[features_to_scale])
test_scaled = scaler.transform(test_data[features_to_scale])

# 4. تحويل النتائج إلى DataFrames
train_scaled_df = pd.DataFrame(train_scaled, columns=features_to_scale, index=train_data.index)
val_scaled_df   = pd.DataFrame(val_scaled,   columns=features_to_scale, index=val_data.index)
test_scaled_df  = pd.DataFrame(test_scaled,  columns=features_to_scale, index=test_data.index)

print("Scaling completed cleanly across Train, Validation, and Test sets!")
print('=' * 65)
print('Chronological Data Split & Timeline Verification:')
print('=' * 65)
print(
    f'Train Period:      {train_data.index.min()} to {train_data.index.max()}  (Rows: {len(train_data)})'
)
print(
    f'Validation Period: {val_data.index.min()} to {val_data.index.max()}  (Rows: {len(val_data)})'
)
print(
    f'Test Period:       {test_data.index.min()} to {test_data.index.max()}  (Rows: {len(test_data)})'
)
print('=' * 65)
print('Track C Scaling completed cleanly without Data Leakage!')

Scaling completed cleanly across Train, Validation, and Test sets!
Chronological Data Split & Timeline Verification:
Train Period:      2015-01-01 00:00:00+00:00 to 2016-12-31 23:00:00+00:00  (Rows: 17544)
Validation Period: 2017-01-01 00:00:00+00:00 to 2017-12-31 23:00:00+00:00  (Rows: 8760)
Test Period:       2018-01-01 00:00:00+00:00 to 2018-12-31 22:00:00+00:00  (Rows: 8759)
Track C Scaling completed cleanly without Data Leakage!


## 5. Sliding Window Integrity Verification (No Data Leakage Example)

Create a hand-crafted small example window (Lookback = 168 hours, Horizon = 24 hours) to explicitly confirm that all input timestamps strictly precede target timestamps.

In [9]:
# 1. إعداد المعلمات المعيارية للنوافذ الزمنية
lookback_hours = 168  # 7 أيام في الماضي كـ inputs (X)
horizon_hours = 24  # 24 ساعة في المستقبل كـ targets (y)

# أخذ عينة تجريبية من بداية فترة التحقق (Validation 2017)
sample_start_idx = 0

# استخراج نطاق تواريخ المدخلات ونطاق تواريخ الأهداف
x_dates = val_scaled_df.index[
    sample_start_idx : sample_start_idx + lookback_hours
]
y_dates = val_scaled_df.index[
    sample_start_idx
    + lookback_hours : sample_start_idx
    + lookback_hours
    + horizon_hours
]

# 2. فحص الأوقات برمجياً والتأكد من عدم وجود تداخل
last_input_date = x_dates.max()
first_target_date = y_dates.min()

assert (
    last_input_date < first_target_date
), "Data Leakage Warning: Input dates overlap with Target dates!"

print("=" * 65)
print("Sliding Window Integrity Check (Example Window):")
print("=" * 65)
print(f"Inputs  (X) Time Range : {x_dates.min()} --> {last_input_date}")
print(f"Targets (y) Time Range : {first_target_date} --> {y_dates.max()}")
print("=" * 65)
print(
    "VERIFIED: All input timestamps strictly precede target timestamps (Leak-Free Window confirmed)!"
)

Sliding Window Integrity Check (Example Window):
Inputs  (X) Time Range : 2017-01-01 00:00:00+00:00 --> 2017-01-07 23:00:00+00:00
Targets (y) Time Range : 2017-01-08 00:00:00+00:00 --> 2017-01-08 23:00:00+00:00
VERIFIED: All input timestamps strictly precede target timestamps (Leak-Free Window confirmed)!
